# Chain

In [1]:
import os
from openai import OpenAI

from dotenv import load_dotenv, find_dotenv
load_dotenv(override=True)

True

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Write me a poem about {topic}"
)

model = ChatOpenAI(model="deepseek-chat", 
                  api_key=os.getenv("DEEPSEEK_API_KEY"),
                  base_url=os.getenv("DEEPSEEK_BASE_URL"))
output_parser = StrOutputParser()
 
#Create a simple chain
chain = prompt | model | output_parser
 
chain

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Write me a poem about {topic}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000002725FD452B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002721244AC60>, root_client=<openai.OpenAI object at 0x0000027213B99010>, root_async_client=<openai.AsyncOpenAI object at 0x0000027213B974D0>, model_name='deepseek-chat', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com', openai_proxy=None, stream_chunk_timeout=120.0)
| StrOutputParser()

In [4]:
response = chain.invoke({"topic": "star and space"})
print(response)

The star awakens in the velvet night,
A single thread of everlasting light.
It does not ask for witness or for name,
But burns in silence on the endless frame.

It sees the slow dance of the turning spheres,
The cosmic dust that falls like silent tears.
It feels the pull of gravity’s deep hand,
The architect of nothing and of land.

A traveler drifts between the worlds apart,
A fragile vessel with a human heart.
He looks upon the star, a distant sun,
And wonders if its journey is begun.

Does it remember its own violent birth?
The clashing gas that formed its place on earth?
Or does it simply glow without a past,
A present moment that will always last?

The space between them, cold and vast and deep,
Holds secrets that the waking never keep.
It whispers of a time before the ground,
When all the light was gathered in a sound.

So star and man, two beings in the void,
By distance bound, yet never quite destroyed.
One burns in faith, one watches from afar—
One is the question, one the fal

In [27]:
chain.batch([{"topic": "sun"}, {"topic": "moon"}])  # Answer multiple questions simultaneously

["The sun, a coin of molten gold,\nIs flung where night was, dark and cold.\nA burning chalice, raised on high,\nTo spill its fervor on the sky.\n\nIt wakes the lark with golden string,\nAnd paints the rose's opening.\nIt licks the dew from every leaf,\nAnd loosens morning from its grief.\n\nIt dries the shroud of misty lawn\nThat clung so close to kiss the dawn.\nA patient, slow, and ancient hand\nThat draws the shadows from the land.\n\nAnd when the weary world is stilled,\nAnd all its harsher edges filled,\nIt sinks, a sudden, bleeding bloom,\nTo gild a lonely, waiting room.\n\nIt paints the clouds in rose and fire,\nA final chord from heaven's lyre.\nIt tucks the day into the west,\nAnd grants the aching world its rest.\n\nA furnace and a gentle guide,\nA terror and a friend beside.\nA promise that the deeps of night\nWill always, always yield to light.",
 "The sky, a field of deepest blue,\nA farmer, dusk, his work is through.\nHe hangs his lantern, soft and white,\nTo guard the s

# More complex chain with database search

## Build vector database

In [10]:
# from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.runnables import RunnableParallel

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = DocArrayInMemorySearch.from_texts(
    ["My package tracking link shows it is still in transit.",
     "We offer a 30-day return policy on all unworn clothing items.",
     "To exchange an item for a different size, please use our portal.",
     "International shipping typically takes between 7 to 14 business days.",
     "Enter the promo code WELCOME15 at checkout for a discount."],
    embedding=embeddings
    )   # Create a vector database

retriever = vectorstore.as_retriever()  # Create a retriever

c:\Users\60410\Desktop\claude code\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7233.50it/s]


In [11]:
retriever.invoke("How can I get a discount?")

[Document(metadata={}, page_content='Enter the promo code WELCOME15 at checkout for a discount.'),
 Document(metadata={}, page_content='We offer a 30-day return policy on all unworn clothing items.'),
 Document(metadata={}, page_content='To exchange an item for a different size, please use our portal.'),
 Document(metadata={}, page_content='My package tracking link shows it is still in transit.')]

## Provide user's questions with runablemap

In [12]:
template = """Answer the question based only on the following context:
{context}
Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [13]:
inputs = RunnableParallel({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
})
 
chain = inputs | prompt | model | output_parser
 
chain

{
  context: RunnableLambda(...),
  question: RunnableLambda(...)
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\nQuestion: {question}\n'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000002723F50B080>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000272406D5070>, root_client=<openai.OpenAI object at 0x000002723F5092B0>, root_async_client=<openai.AsyncOpenAI object at 0x000002724061F530>, model_name='deepseek-chat', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_

In [ ]:
# LLM summaries the most relevant answer.
chain.invoke({"question": "How can I get a discount?"})

'Enter the promo code WELCOME15 at checkout for a discount.'

In [ ]:
# RunnableMap provides four documents related to the problem for context variables.
inputs.invoke({"question": "How can I get a discount?"})

{'context': [Document(metadata={}, page_content='Enter the promo code WELCOME15 at checkout for a discount.'),
  Document(metadata={}, page_content='We offer a 30-day return policy on all unworn clothing items.'),
  Document(metadata={}, page_content='To exchange an item for a different size, please use our portal.'),
  Document(metadata={}, page_content='My package tracking link shows it is still in transit.')],
 'question': 'How can I get a discount?'}

# Bind
Function call function

In [21]:
import json, requests

def get_current_weather(location, unit="Celsius"):
    """Get the current weather in a given location"""
    '''
    url = "https://devapi.qweather.com/v7/weather/now"
    params = {
        "location": location,     
        "key": "HEWEATHER_API_KEY", 
        "unit": "m" if unit == "celsius" else "i" 
    }
    response = requests.get(url, params=params)
    data = response.json()
    '''
    weather_info = {
        "location": location, 
        "temperature":  "20",    # data["now"]["temp"]
        "unit": unit, 
        "forecast": ["sunny", "windy", "cloudy", "rainy"], 
    }
    return json.dumps(weather_info)

functions = [   ## LLM can only read text: function name, description, parameter type and return
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {
                    "type": "string", 
                    "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
]

In [23]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}")
    ]
)

model = ChatOpenAI(model="deepseek-chat", 
                  api_key=os.getenv("DEEPSEEK_API_KEY"),
                  base_url=os.getenv("DEEPSEEK_BASE_URL")).bind(functions=functions)
# Model will evaluate whether the prompt can be solved by functions

runnable = prompt | model
 
runnable

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| _ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000027213C2BDD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002721249AC60>, root_client=<openai.OpenAI object at 0x00000272121E0E60>, root_async_client=<openai.AsyncOpenAI object at 0x0000027213C29730>, model_name='deepseek-chat', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com', openai_proxy=None, stream_chunk_timeout=120.0), kwargs={'functions': [{'name': 'get_current_weather', 'description': 'Get

In [24]:
response = runnable.invoke({"input": "What's the weather in San Diego？"})
response

AIMessage(content="I don't have real-time access to current weather data. To get the current weather in San Diego, I recommend checking a reliable weather website or app, such as weather.com, AccuWeather, or the National Weather Service.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 12, 'total_tokens': 58, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 12}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '50d76771-7a1c-45b2-b21b-a77d1359cf2d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f322a-9569-7220-977d-1f887860d61c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 46, 'total_tokens': 58, 'input_token_details': {'cache_read': 0}, 'output_token_details':